In [1]:
# Exact deps needed by his scripts
!pip install -q transformers peft accelerate bitsandbytes datasets scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.7 MB/s eta 0:00:00


In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import joblib

# His exact env var — on Kaggle H100 there's 1 GPU so we use "0" only
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# His TEMP_DIR pattern, adapted for Kaggle
TEMP_DIR = "/kaggle/working/tmp"
os.makedirs(TEMP_DIR, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

GPU: NVIDIA H100 80GB HBM3
VRAM: 85.0 GB


In [3]:
# His exact line, path adapted for Kaggle
train = pd.read_csv('/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv')
test  = pd.read_csv('/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")
print(train.head())

Train shape: (36696, 7)
Test  shape: (3, 5)
   row_id  QuestionId                                       QuestionText  \
0       0       31772  What fraction of the shape is not shaded? Give...   
1       1       31772  What fraction of the shape is not shaded? Give...   
2       2       31772  What fraction of the shape is not shaded? Give...   
3       3       31772  What fraction of the shape is not shaded? Give...   
4       4       31772  What fraction of the shape is not shaded? Give...   

           MC_Answer                                 StudentExplanation  \
0  \( \frac{1}{3} \)                  0ne third is equal to tree nineth   
1  \( \frac{1}{3} \)  1 / 3 because 6 over 9 is 2 thirds and 1 third...   
2  \( \frac{1}{3} \)  1 3rd is half of 3 6th, so it is simplee to un...   
3  \( \frac{1}{3} \)        1 goes into everything and 3 goes into nine   
4  \( \frac{1}{3} \)                    1 out of every 3 isn't coloured   

       Category Misconception  
0  True_Correct 

In [4]:
# His exact 3 lines:
train.Misconception = train.Misconception.fillna('NA')
train['target'] = train.Category + ":" + train.Misconception

le = LabelEncoder()
train['label'] = le.fit_transform(train['target'])
n_classes = len(le.classes_)

print(f"Train shape: {train.shape} with {n_classes} target classes")
print(train.head())

Train shape: (36696, 9) with 65 target classes
   row_id  QuestionId                                       QuestionText  \
0       0       31772  What fraction of the shape is not shaded? Give...   
1       1       31772  What fraction of the shape is not shaded? Give...   
2       2       31772  What fraction of the shape is not shaded? Give...   
3       3       31772  What fraction of the shape is not shaded? Give...   
4       4       31772  What fraction of the shape is not shaded? Give...   

           MC_Answer                                 StudentExplanation  \
0  \( \frac{1}{3} \)                  0ne third is equal to tree nineth   
1  \( \frac{1}{3} \)  1 / 3 because 6 over 9 is 2 thirds and 1 third...   
2  \( \frac{1}{3} \)  1 3rd is half of 3 6th, so it is simplee to un...   
3  \( \frac{1}{3} \)        1 goes into everything and 3 goes into nine   
4  \( \frac{1}{3} \)                    1 out of every 3 isn't coloured   

       Category Misconception           targe

In [5]:
# His exact block, copy-pasted verbatim:
idx = train.apply(lambda row: row.Category.split('_')[0], axis=1) == 'True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId', 'MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c', ascending=False)
correct = correct.drop_duplicates(['QuestionId'])
correct = correct[['QuestionId', 'MC_Answer']]
correct['is_correct'] = 1  # Mark these as correct answers

# His exact merge line:
train = train.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
train.is_correct = train.is_correct.fillna(0)

# Also apply to test set (same logic needed at inference)
test = test.merge(correct, on=['QuestionId', 'MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)

print("is_correct distribution in train:")
print(train.is_correct.value_counts())

is_correct distribution in train:
is_correct
1.0    20464
0.0    16232
Name: count, dtype: int64


In [6]:
# His exact function from Qwen3-4B-0.945.py:
def format_input_v2(row):
    x = "Yes"
    if not row['is_correct']:
        x = "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Is Correct Answer: {x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

train['text'] = train.apply(format_input_v2, axis=1)

print("\nExample prompt for our LLM (after refinement):")
print(train.text.values[0])


Example prompt for our LLM (after refinement):
Question: What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them are shaded.]
Answer: \( \frac{1}{3} \)
Is Correct Answer: Yes
Student Explanation: 0ne third is equal to tree nineth


In [7]:
# Find classes with only 1 row
counts = train['label'].value_counts()
singletons = counts[counts == 1].index

# Display the rows so you can see what they are
problem_rows = train[train['label'].isin(singletons)]
print(problem_rows[['label', 'target', 'QuestionText', 'StudentExplanation']])

       label                                   target  \
15189     54          True_Misconception:Not_variable   
20364     61  True_Misconception:Whole_numbers_larger   
21212     51      True_Misconception:Longer_is_bigger   
23580     38         True_Misconception:Adding_across   
35167     40             True_Misconception:Base_rate   

                                            QuestionText  \
15189        \( 2 y=24 \) What is the value of \( y \) ?   
20364                      Which number is the greatest?   
21212                      Which number is the greatest?   
23580                     \( \frac{1}{3}+\frac{2}{5}= \)   
35167  It takes \( 3 \) people a total of \( 192 \) h...   

                                      StudentExplanation  
15189  i think because 2*12is 24 so that's why im sor...  
20364                   6 is bigger than decimal numbers  
21212  d because it has a 6 the same as a but it also...  
23580  1/3 is 5/15\r\n2/5 is 6/15\r\n5+6 is 11 so the...  
3

In [8]:
# Find singleton classes
counts = train['label'].value_counts()
singletons = counts[counts == 1].index

# Duplicate those rows so stratify works (each class needs min 2 samples)
singleton_rows = train[train['label'].isin(singletons)]
train = pd.concat([train, singleton_rows], ignore_index=True)

# Verify — should be 0 now
counts_after = train['label'].value_counts()
still_singleton = counts_after[counts_after == 1]
print(f"Singletons before fix: {len(singletons)}")
print(f"Singletons after fix:  {len(still_singleton)}")  # should print 0
print(f"Train size after duplication: {len(train)}")

Singletons before fix: 5
Singletons after fix:  0
Train size after duplication: 36701


In [9]:
# Hunyuan script uses val_split=0.1 — we follow the same
train_df, val_df = train_test_split(
    train,
    test_size=0.10,
    random_state=42,
    stratify=train['label']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

Train: 33,030 | Val: 3,671


In [10]:
# His exact Model_Name from Qwen3-4B-0.945.py
Model_Name = "Qwen/Qwen3-4B"   # public version of Qwen3-4B (no HF token needed)

# His exact model load from Qwen3-4B-0.945.py:
model = AutoModelForSequenceClassification.from_pretrained(
    Model_Name,
    num_labels=n_classes,
    device_map="auto",          # "balanced" in his script was for 2-GPU Colab; "auto" for Kaggle H100
    cache_dir=TEMP_DIR,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    Model_Name,
    cache_dir=TEMP_DIR,
    trust_remote_code=True
)

# His exact LoRA config from Qwen3-4B-0.945.py:
lora_config = LoraConfig(
    r=512,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.3,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"]   # his exact line
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# His exact line:
model = model.to(dtype=torch.bfloat16)

print(next(model.parameters()).dtype)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-4B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

torch.bfloat16
trainable params: 991,070,720 || all params: 5,013,705,216 || trainable%: 19.7672


In [11]:
# His exact 3-line block, present in every single script:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

print("Pad token:", tokenizer.pad_token)
print("Vocab size after resize:", len(tokenizer))

Pad token: [PAD]
Vocab size after resize: 151670


In [12]:
# His exact COLS and tokenize function:
COLS = ['text', 'label']

train_df_clean = train_df[COLS].copy()
train_df_clean['label'] = train_df_clean['label'].astype(np.int64)
train_df_clean = train_df_clean.reset_index(drop=True)

val_df_clean = val_df[COLS].copy()
val_df_clean['label'] = val_df_clean['label'].astype(np.int64)
val_df_clean = val_df_clean.reset_index(drop=True)

# His exact Dataset creation:
train_ds = Dataset.from_pandas(train_df_clean, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df_clean,   preserve_index=False)

# His exact tokenize function:
def tokenize(batch):
    """Tokenizes a batch of text inputs."""
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=['text'])

print("Train tokens sample length:", len(train_ds[0]['input_ids']))
print("Val size:", len(val_ds))

Map:   0%|          | 0/33030 [00:00<?, ? examples/s]

Map:   0%|          | 0/3671 [00:00<?, ? examples/s]

Train tokens sample length: 57
Val size: 3671


In [13]:
# His exact compute_map3 function, copy-pasted verbatim:
def compute_map3(eval_pred):
    """
    Computes Mean Average Precision at 3 (MAP@3) for evaluation.
    """
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

    # Get top 3 predicted class indices for each sample
    top3 = np.argsort(-probs, axis=1)[:, :3]

    # Check if the true label is within the top 3 predictions
    match = (top3 == labels[:, None])  # Create a boolean array indicating matches

    map3 = 0.0
    for i in range(len(labels)):
        if match[i, 0]:    # If true label is in the 1st prediction
            map3 += 1.0
        elif match[i, 1]:  # If true label is in the 2nd prediction
            map3 += 1.0 / 2
        elif match[i, 2]:  # If true label is in the 3rd prediction
            map3 += 1.0 / 3

    return {"map@3": map3 / len(labels)}  # Average MAP@3 over all samples

In [14]:
os.makedirs(f"{TEMP_DIR}/training_output/", exist_ok=True)
os.makedirs(f"{TEMP_DIR}/logs/", exist_ok=True)

# His exact TrainingArguments from Qwen3-4B-0.945.py
# (do_eval=False → changed to True so we can see MAP@3 live)
training_args = TrainingArguments(
    output_dir=f"{TEMP_DIR}/training_output/",
    do_train=True,
    do_eval=True,                    # enabled so MAP@3 prints each epoch
    save_strategy="no",              # his exact value
    eval_strategy="epoch",           # added to show accuracy
    num_train_epochs=3,              # his exact value
    per_device_train_batch_size=8,   # his exact value
    per_device_eval_batch_size=16,
    learning_rate=2e-4,              # his exact value
    logging_dir=f"{TEMP_DIR}/logs/",
    logging_steps=100,               # his exact value
    gradient_accumulation_steps=4,   # his exact value
    remove_unused_columns=False,     # his exact value
    bf16=True,                       # his exact value
    fp16=False,                      # his exact value
    report_to="none",                # his exact value
    warmup_ratio=0.1,                # his exact value
    lr_scheduler_type="cosine",      # his exact value
    dataloader_drop_last=True,       # his exact value
    dataloader_pin_memory=False,     # his exact value
    gradient_checkpointing=True,     # his exact value
)

print("Effective batch size:", training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Effective batch size: 32


In [15]:
# His exact data_collator and Trainer setup:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,           # added for MAP@3 display
    processing_class=tokenizer,    # his exact kwarg (not tokenizer=)
    data_collator=data_collator,
    compute_metrics=compute_map3,  # added so MAP@3 prints each epoch
)

# His exact call:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Map@3
1,1.556661,0.353086,0.934634
2,0.895638,0.301809,0.942185
3,0.406894,0.356532,0.944278


TrainOutput(global_step=3096, training_loss=1.5804916130479916, metrics={'train_runtime': 4072.2611, 'train_samples_per_second': 24.333, 'train_steps_per_second': 0.76, 'total_flos': 2.8811528054614426e+17, 'train_loss': 1.5804916130479916, 'epoch': 3.0})

In [16]:
from scipy.special import softmax as sp_softmax

val_output  = trainer.predict(val_ds)
probs_val   = sp_softmax(val_output.predictions, axis=1)
top3_val    = np.argsort(-probs_val, axis=1)[:, :3]
true_labels = val_df['label'].values

# Align lengths since drop_last=True cuts last batch during predict
n = min(len(top3_val), len(true_labels))
top3_val    = top3_val[:n]
true_labels = true_labels[:n]

# MAP@3
map3_score = 0.0
for i in range(n):
    for rank in range(3):
        if top3_val[i, rank] == true_labels[i]:
            map3_score += 1.0 / (rank + 1)
            break
map3_score /= n

top1_acc = np.mean(top3_val[:, 0] == true_labels)
top3_acc = np.mean([true_labels[i] in top3_val[i] for i in range(n)])

print("=" * 45)
print(f"  MAP@3  (official metric):  {map3_score:.4f}")
print(f"  Top-1  Accuracy:           {top1_acc:.4f}")
print(f"  Top-3  Accuracy:           {top3_acc:.4f}")
print("=" * 45)

# Per-family breakdown
val_df_copy = val_df.iloc[:n].copy().reset_index(drop=True)
val_df_copy['family'] = val_df_copy['Category'].apply(lambda x: x.split('_')[0])

for fam in ['True', 'False']:
    mask = (val_df_copy['family'] == fam).values  # size n, matches top3_val
    if mask.sum() == 0:
        continue
    fam_top3  = top3_val[mask]
    fam_true  = true_labels[mask]
    fam_score = 0.0
    for i in range(len(fam_true)):
        for rank in range(3):
            if fam_top3[i, rank] == fam_true[i]:
                fam_score += 1.0 / (rank + 1)
                break
    fam_score /= len(fam_true)
    print(f"  MAP@3 [{fam:5s}]: {fam_score:.4f}   (n={mask.sum():,})")

  MAP@3  (official metric):  0.9443
  Top-1  Accuracy:           0.8925
  Top-3  Accuracy:           0.9981
  MAP@3 [True ]: 0.9490   (n=2,041)
  MAP@3 [False]: 0.9383   (n=1,623)


In [17]:
# His exact save block from Qwen3-4B-0.945.py, path adapted for Kaggle:
print(f"Saving best model to best")
trainer.save_model(f"/kaggle/working/qwen3_4b/best")
_ = joblib.dump(le, f"/kaggle/working/qwen3_4b/label_encoder.joblib")

print("Saved model to /kaggle/working/qwen3_4b/best")
print("Saved label encoder to /kaggle/working/qwen3_4b/label_encoder.joblib")

Saving best model to best


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Saved model to /kaggle/working/qwen3_4b/best
Saved label encoder to /kaggle/working/qwen3_4b/label_encoder.joblib


In [18]:
# # Build test dataset — NO dummy label column
# test['text'] = test.apply(format_input_v2, axis=1)

# test_df_clean = test[['text']].copy().reset_index(drop=True)
# # DO NOT add label column
# test_ds = Dataset.from_pandas(test_df_clean, preserve_index=False)
# test_ds = test_ds.map(tokenize, batched=True, remove_columns=['text'])

# # Predict
# test_output = trainer.predict(test_ds)

# # Robustly unwrap — handles nested tuples/lists AND 0-d object arrays
# preds_raw = test_output.predictions
# while isinstance(preds_raw, (tuple, list)):
#     preds_raw = preds_raw[0]

# preds_raw = np.asarray(preds_raw)

# # Key fix: 0-d object arrays wrap the real array inside — keep peeling
# while preds_raw.ndim == 0:
#     preds_raw = np.asarray(preds_raw.item())

# preds_raw = preds_raw.astype(np.float32)

# # Guard for single-sample edge case
# if preds_raw.ndim == 1:
#     preds_raw = preds_raw.reshape(1, -1)

# probs_test = sp_softmax(preds_raw, axis=1)
# top3_test  = np.argsort(-probs_test, axis=1)[:, :3]

# # Decode back to "Category:Misconception" strings
# preds = [" ".join([le.classes_[j] for j in row]) for row in top3_test]

# submission = pd.DataFrame({
#     'row_id': test['row_id'],
#     'MisconceptionId': preds
# })
# submission.to_csv("/kaggle/working/submission.csv", index=False)
# print("Submission saved.")
# print(submission.head())